In [7]:
import pandas as pd
import editdistance
import sys
sys.path.insert(0, '..')
from src.normalisation import normalize

# Charger 200 lignes du val set
val_df = pd.read_csv('../dataset_nlp/splits/val.csv').sample(200, random_state=42)
print(f"Échantillon : {len(val_df)} lignes")
print(f"Colonnes : {val_df.columns.tolist()}")
print(val_df['text'].iloc[0])

Échantillon : 200 lignes
Colonnes : ['text', 'image_path', 'language', 'century', 'shelfmark', 'project', 'source_corpus']
Loys, par la grace de Dieu roy de France. Savoir faisons, à tous, presens et advenir, nous avoir receu l’umble supplicacion


In [8]:
import numpy as np

def cer(pred: str, gt: str) -> float:
    if len(gt) == 0:
        return 0.0
    return editdistance.eval(pred, gt) / len(gt)

results = []
for _, row in val_df.iterrows():
    original = str(row['text'])
    normalized = normalize(original)
    c = cer(normalized, original)
    results.append({
        'original': original,
        'normalized': normalized,
        'cer_modification': c,
        'changed': original != normalized
    })

df_results = pd.DataFrame(results)
n_changed = df_results['changed'].sum()
cer_mean = df_results['cer_modification'].mean()

print(f"Lignes modifiées : {n_changed}/200 ({n_changed/2:.1f}%)")
print(f"Taux de modification moyen (CER) : {cer_mean:.4f} ({cer_mean*100:.2f}%)")
print(f"\nExemples de modifications :")
changed = df_results[df_results['changed']][['original','normalized']].head(5)
for _, r in changed.iterrows():
    print(f"  AVANT : {r['original']}")
    print(f"  APRÈS : {r['normalized']}")
    print()

Lignes modifiées : 128/200 (64.0%)
Taux de modification moyen (CER) : 0.0261 (2.61%)

Exemples de modifications :
  AVANT : Loys, par la grace de Dieu roy de France. Savoir faisons, à tous, presens et advenir, nous avoir receu l’umble supplicacion
  APRÈS : Loys, par la grace de Dieu roy de France. Savoir faisons, à tous, presens et advenir, nous avoir receu l’umble supplicacion

  AVANT : alèrent où ce faisoit ledit debat, et y alant icelluy suppliant rencontra ung nommé Anthoine de France auquel il demanda
  APRÈS : alèrent où ce faisoit ledit debat, et y alant icelluy suppliant rencontra ung nommé Anthoine de France auquel il demanda

  AVANT : Loys, par la grace de Dieu roy de France. Savoir faisons à tous, presens et avenir, nous avoir receue l’umble supplicacion
  APRÈS : Loys, par la grace de Dieu roy de France. Savoir faisons à tous, presens et avenir, nous avoir receue l’umble supplicacion

  AVANT : Et quant vous m’eussiez demandé du poisson, je vous en eusse bien donné,

In [9]:
from src.normalisation import normalize_uv, normalize_ij, expand_abbreviations, normalize_unicode

# Debug sur les cas problématiques
tests = ["je", "Jehan", "desrober"]
for t in tests:
    a1 = expand_abbreviations(t)
    a2 = normalize_unicode(a1)
    a3 = normalize_uv(a2)
    a4 = normalize_ij(a3)
    print(f"{t!r} → abbr={a1!r} → nfd={a2!r} → uv={a3!r} → ij={a4!r}")

'je' → abbr='je' → nfd='je' → uv='je' → ij='ie'
'Jehan' → abbr='Jehan' → nfd='Jehan' → uv='Jehan' → ij='iehan'
'desrober' → abbr='desireober' → nfd='desireober' → uv='desireober' → ij='desireober'


In [10]:
import importlib
import src.normalisation
importlib.reload(src.normalisation)
from src.normalisation import normalize, normalize_uv, normalize_ij, expand_abbreviations, normalize_unicode

# Vérification rapide
print(expand_abbreviations("desrober"))  # doit rester 'desrober'
print(normalize_ij("je vous"))           # doit rester 'je vous'
print(normalize_ij("jour"))              # doit rester 'jour' (j devant o... attends)

desrober
je vous
iour


In [11]:
results = []
for _, row in val_df.iterrows():
    original = str(row['text'])
    normalized = normalize(original)
    c = cer(normalized, original)
    results.append({
        'original': original,
        'normalized': normalized,
        'cer_modification': c,
        'changed': original != normalized
    })

df_results = pd.DataFrame(results)
n_changed = df_results['changed'].sum()
cer_mean = df_results['cer_modification'].mean()

print(f"Lignes modifiées : {n_changed}/200 ({n_changed/2:.1f}%)")
print(f"Taux de modification moyen (CER) : {cer_mean:.4f} ({cer_mean*100:.2f}%)")
print(f"\nExemples de modifications :")
changed = df_results[df_results['changed']][['original','normalized']].head(5)
for _, r in changed.iterrows():
    print(f"  AVANT : {r['original']}")
    print(f"  APRÈS : {r['normalized']}")
    print()

Lignes modifiées : 122/200 (61.0%)
Taux de modification moyen (CER) : 0.0251 (2.51%)

Exemples de modifications :
  AVANT : Loys, par la grace de Dieu roy de France. Savoir faisons, à tous, presens et advenir, nous avoir receu l’umble supplicacion
  APRÈS : Loys, par la grace de Dieu roy de France. Savoir faisons, à tous, presens et advenir, nous avoir receu l’umble supplicacion

  AVANT : alèrent où ce faisoit ledit debat, et y alant icelluy suppliant rencontra ung nommé Anthoine de France auquel il demanda
  APRÈS : alèrent où ce faisoit ledit debat, et y alant icelluy suppliant rencontra ung nommé Anthoine de France auquel il demanda

  AVANT : Loys, par la grace de Dieu roy de France. Savoir faisons à tous, presens et avenir, nous avoir receue l’umble supplicacion
  APRÈS : Loys, par la grace de Dieu roy de France. Savoir faisons à tous, presens et avenir, nous avoir receue l’umble supplicacion

  AVANT : Et quant vous m’eussiez demandé du poisson, je vous en eusse bien donné,

In [12]:
import json
from datetime import datetime

entry = {
    "step": "nlp_normalisation_regles",
    "date": "2026-06-18",
    "description": (
        "Normalisation orthographique par règles sur 200 lignes val set. "
        "Pipeline: abréviations → NFD → u/v → i/j → ponctuation."
    ),
    "n_lignes": 200,
    "lignes_modifiees": int(n_changed),
    "pct_modifiees": round(n_changed/2, 1),
    "cer_modification_moyen": round(cer_mean, 4),
    "seuil_validation": 0.10,
    "statut": "VALIDE (CER < 10%)",
    "notes": (
        "Règles appliquées: Unicode NFD, abréviations médiévales (ñ→nn, ꝵ→rum), "
        "u initial devant voyelle → v, j initial devant a/o/u → i. "
        "Faux positifs corrigés: sr→sire supprimé, j devant e/i conservé."
    )
}

with open("../experiments/journal.jsonl", "a", encoding="utf-8") as f:
    f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print("✅ Entrée journal ajoutée")

✅ Entrée journal ajoutée


In [1]:
from transformers import pipeline

print("Chargement du modèle CamemBERT-NER...")
ner_pipeline = pipeline(
    "ner",
    model="Jean-Baptiste/camembert-ner",
    aggregation_strategy="simple",
    device=0  # GPU
)

# Test rapide
test = "Le roi Louis de France signa une charte à Paris le 15 mars 1420."
result = ner_pipeline(test)
print(f"\nTest NER sur : {test!r}")
for ent in result:
    print(f"  {ent['entity_group']:6} | {ent['word']:20} | score={ent['score']:.3f}")

/root/htr-medieval-manuscripts-XIVe/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Chargement du modèle CamemBERT-NER...


Loading weights: 100%|███████████████████████████████████████████████| 199/199 [00:00<00:00, 1864.10it/s]



Test NER sur : 'Le roi Louis de France signa une charte à Paris le 15 mars 1420.'
  PER    | Louis de France      | score=0.995
  LOC    | Paris                | score=0.988


In [3]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import importlib
import src.normalisation
importlib.reload(src.normalisation)
from src.normalisation import normalize

val_df = pd.read_csv('../dataset_nlp/splits/val.csv').sample(200, random_state=42)
val_df['text_normalized'] = val_df['text'].apply(lambda t: normalize(str(t)))

print(f"Corpus : {len(val_df)} lignes normalisées")
print(f"Exemple : {val_df['text_normalized'].iloc[0][:100]}")

print("\nApplication NER...")
all_entities = []
for idx, row in val_df.iterrows():
    text = row['text_normalized']
    try:
        entities = ner_pipeline(text[:512])
        for ent in entities:
            all_entities.append({
                'text_idx': idx,
                'word': ent['word'],
                'entity': ent['entity_group'],
                'score': ent['score'],
                'source': row.get('source_corpus', '?'),
                'century': row.get('century', '?')
            })
    except Exception as e:
        pass

df_ner = pd.DataFrame(all_entities)
print(f"\nEntités détectées : {len(df_ner)}")
print(f"\nDistribution par type :")
print(df_ner['entity'].value_counts())
print(f"\nScore moyen par type :")
print(df_ner.groupby('entity')['score'].mean().round(3))
print(f"\nExemples PER :")
print(df_ner[df_ner['entity']=='PER']['word'].head(10).tolist())
print(f"\nExemples LOC :")
print(df_ner[df_ner['entity']=='LOC']['word'].head(10).tolist())

Corpus : 200 lignes normalisées
Exemple : Loys, par la grace de Dieu roy de France. Savoir faisons, à tous, presens et advenir, nous avoir re

Application NER...


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset



Entités détectées : 215

Distribution par type :
entity
PER     143
LOC      54
MISC     14
ORG       4
Name: count, dtype: int64

Score moyen par type :
entity
LOC     0.878
MISC    0.757
ORG     0.789
PER     0.895
Name: score, dtype: float32

Exemples PER :
['roy de France', 'Anthoine de France', 'Dieu roy de France', 'ulx', 'Thabouril', 'Jehan Rousseau', 'Simonne', 'Pouffe', 'Maynner', 'Pierre Norrisson']

Exemples LOC :
['Lespau', 'ioye', 'Calabre', 'Angiers', 'Rasillé', 'Braye', 'aigne', 'espaulles', 'la Rochelle', 'ioye']


In [4]:
# Évaluation approximative : on vérifie manuellement les 20 premières PER et LOC
# et on calcule une précision estimée

sample_per = df_ner[df_ner['entity']=='PER'].head(20)['word'].tolist()
sample_loc = df_ner[df_ner['entity']=='LOC'].head(20)['word'].tolist()

print("=== 20 premières entités PER ===")
for i, w in enumerate(sample_per):
    print(f"  {i+1:2}. {w!r}")

print("\n=== 20 premières entités LOC ===")
for i, w in enumerate(sample_loc):
    print(f"  {i+1:2}. {w!r}")

# Statistiques globales
print(f"\n=== Résumé NER zéro-shot ===")
print(f"Lignes traitées    : {len(val_df)}")
print(f"Entités totales    : {len(df_ner)}")
print(f"Entités/ligne moy  : {len(df_ner)/len(val_df):.2f}")
print(f"Score moyen global : {df_ner['score'].mean():.3f}")
print(f"Score > 0.8        : {(df_ner['score'] > 0.8).sum()} ({(df_ner['score'] > 0.8).mean()*100:.1f}%)")
print(f"\nModèle : Jean-Baptiste/camembert-ner (zéro-shot)")
print(f"Corpus : val set CATMuS+HIMANIS moyen français XIVe-XVe")
print(f"Limitation : pas de type DATE, entités complexes parfois fragmentées")

=== 20 premières entités PER ===
   1. 'roy de France'
   2. 'Anthoine de France'
   3. 'Dieu roy de France'
   4. 'ulx'
   5. 'Thabouril'
   6. 'Jehan Rousseau'
   7. 'Simonne'
   8. 'Pouffe'
   9. 'Maynner'
  10. 'Pierre Norrisson'
  11. 'Caffart'
  12. 'Saint Michiel'
  13. 'Denis du Vergier'
  14. 'Jehan Bouffinière'
  15. 'Denis Bouffinère'
  16. 'supliant'
  17. 'roy ph̾e'
  18. 'ꝯbiẽ'
  19. 'Donne'
  20. 'Pelet'

=== 20 premières entités LOC ===
   1. 'Lespau'
   2. 'ioye'
   3. 'Calabre'
   4. 'Angiers'
   5. 'Rasillé'
   6. 'Braye'
   7. 'aigne'
   8. 'espaulles'
   9. 'la Rochelle'
  10. 'ioye'
  11. 'cuer'
  12. 'veneour'
  13. 'voulans'
  14. 'prins'
  15. 'forche'
  16. 'Thouars'
  17. 'Tours'
  18. 'bõne'
  19. 'mauuaise'
  20. 'coste'

=== Résumé NER zéro-shot ===
Lignes traitées    : 200
Entités totales    : 215
Entités/ligne moy  : 1.07
Score moyen global : 0.880
Score > 0.8        : 164 (76.3%)

Modèle : Jean-Baptiste/camembert-ner (zéro-shot)
Corpus : val set CA

In [5]:
import json

entry = {
    "step": "nlp_ner_zeroshot_camembert",
    "date": "2026-06-18",
    "description": (
        "NER zéro-shot avec Jean-Baptiste/camembert-ner sur 200 lignes "
        "val set normalisées. Pas de fine-tuning, évaluation exploratoire."
    ),
    "modele": "Jean-Baptiste/camembert-ner",
    "n_lignes": 200,
    "entites_detectees": int(len(df_ner)),
    "distribution": df_ner['entity'].value_counts().to_dict(),
    "score_moyen": round(float(df_ner['score'].mean()), 3),
    "score_gt_08_pct": round(float((df_ner['score'] > 0.8).mean() * 100), 1),
    "precision_estimee_per": 0.60,
    "precision_estimee_loc": 0.35,
    "precision_globale_estimee": 0.50,
    "limitations": [
        "Pas de type DATE supporté par le modèle",
        "Faux positifs LOC sur mots médiévaux (ioye, espaulles, cuer)",
        "Abréviations non résolues (ꝯbiẽ, roy ph̃e) détectées comme PER",
        "Fine-tuning nécessaire pour atteindre F1 > 0.65"
    ],
    "seuil_syllabus": "F1 > 0.65 micro (PER, LOC, DATE)",
    "statut": "EXPLORATOIRE - fine-tuning requis pour validation"
}

with open("../experiments/journal.jsonl", "a", encoding="utf-8") as f:
    f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print("✅ Entrée journal NER ajoutée")

✅ Entrée journal NER ajoutée
